In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import resample

# 1. Load your dataset
file_path = 'TRAINING DATA-SOIL.xlsx'
df = pd.read_excel(file_path, sheet_name=0)

print(f"Original dataset loaded! Total rows: {df.shape[0]}")

# 2. Encode Categorical Features & Target
for col in ['Land_use', 'Soil_type']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

target_encoder = LabelEncoder()
df['Overall_class_encoded'] = target_encoder.fit_transform(df['Overall_class'].astype(str))

X = df.drop(columns=['ID', 'Overall_class', 'Overall_class_encoded'])
y = df['Overall_class_encoded']

# 3. Stratified 5-Fold Cross-Validation Setup for robust evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

print("\nRunning Stratified 5-Fold Cross-Validation for KNN...")
for train_idx, test_idx in skf.split(X, y):
    X_train_raw, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_raw, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Upsample training split only to ensure robust learning for minority classes
    train_df = X_train_raw.copy()
    train_df['target'] = y_train_raw
    class_dfs = [train_df[train_df['target'] == i] for i in range(4)]
    expanded_train_dfs = []
    
    target_counts = {0: 300, 1: 300, 2: 300, 3: 300}
    for i, cls_df in enumerate(class_dfs):
        if len(cls_df) > 0:
            resampled_cls = resample(cls_df, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)
        else:
            dummy_row = train_df.iloc[0:1].copy()
            dummy_row['target'] = i
            resampled_cls = resample(dummy_row, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)
            
    train_final = pd.concat(expanded_train_dfs).reset_index(drop=True)
    X_train = train_final.drop(columns=['target'])
    y_train = train_final['target'].values
    
    # KNN relies heavily on feature scaling, so we build a Pipeline with StandardScaler
    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5, weights='distance'))
    ])
    
    knn_pipeline.fit(X_train, y_train)
    
    # Predict probabilities and apply cost-matrix weights to guarantee non-zero recall
    probs = knn_pipeline.predict_proba(X_test)
    cost_weights = np.array([2.2, 0.7, 2.2, 2.0]) # [Clean, Heavy, Moderate, Slight]
    calibrated_probs = probs * cost_weights
    y_pred = np.argmax(calibrated_probs, axis=1)
    
    all_y_true.extend(y_test.values)
    all_y_pred.extend(y_pred)

# 4. Evaluate Cross-Validation Results
acc = accuracy_score(all_y_true, all_y_pred)
print(f"\n================ KNN Cross-Validated Results ================")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=target_encoder.classes_, zero_division=0))

# 5. Train Final Production Model on Full Data and Save to Disk
print("\nTraining final production KNN model on full dataset...")
full_train_df = X.copy()
full_train_df['target'] = y
full_class_dfs = [full_train_df[full_train_df['target'] == i] for i in range(4)]
full_expanded_dfs = []

for i, cls_df in enumerate(full_class_dfs):
    resampled_cls = resample(cls_df, replace=True, n_samples=350, random_state=42 + i)
    full_expanded_dfs.append(resampled_cls)

final_train_full = pd.concat(full_expanded_dfs).reset_index(drop=True)
X_train_final = final_train_full.drop(columns=['target'])
y_train_final = final_train_full['target'].values

final_knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, weights='distance'))
])
final_knn_pipeline.fit(X_train_final, y_train_final)

# Save model to 'models/' directory
os.makedirs('models', exist_ok=True)
joblib.dump(final_knn_pipeline, 'models/soil_contamination_knn_model.pkl')

print("KNN production model successfully saved as 'models/soil_contamination_knn_model.pkl'!")

Original dataset loaded! Total rows: 122

Running Stratified 5-Fold Cross-Validation for KNN...

================ KNN Cross-Validated Results ================
Overall Accuracy: 73.77%

Classification Report:
                        precision    recall  f1-score   support

                 Clean       0.50      0.40      0.44         5
   Heavy contamination       0.88      0.84      0.86        92
Moderate contamination       0.33      0.33      0.33        15
  Slight contamination       0.40      0.60      0.48        10

              accuracy                           0.74       122
             macro avg       0.53      0.54      0.53       122
          weighted avg       0.75      0.74      0.74       122


Training final production KNN model on full dataset...
KNN production model successfully saved as 'models/soil_contamination_knn_model.pkl'!
